# Day 022 Project: REST API Client

## What You're Building

A `PostsAPI` class that wraps [JSONPlaceholder](https://jsonplaceholder.typicode.com) — a free, always-available fake REST API used by millions of developers for testing. The same patterns you apply here work for any real API: GitHub, Stripe, Notion, Spotify.

## Project Requirements

1. Implement `PostsAPI` with a `requests.Session` and auth headers
2. `get_posts(user_id, limit)` — fetch posts, optionally filtered by user
3. `get_pages(page_size, max_pages)` — collect paginated results
4. `ai_summary(question)` — fetch posts and ask the LLM about them
5. `safe_fetch(endpoint)` — fetch any endpoint safely (no raise)

**You run it, it prints post titles, user stats, and an AI summary. That's the deliverable.**

In [ ]:
import requests
import json
import ollama

## Provided: All Helper Functions

In [ ]:
def get_json(url: str, params: dict | None = None, headers: dict | None = None):
    response = requests.get(url, params=params, headers=headers, timeout=10)
    response.raise_for_status()
    return response.json()


def build_headers(api_key: str, extra: dict | None = None) -> dict:
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }
    if extra:
        headers.update(extra)
    return headers


def paginate_collect(url: str, page_size: int = 10, max_pages: int = 3) -> list[dict]:
    all_results = []
    for page in range(1, max_pages + 1):
        params = {"_page": page, "_limit": page_size}
        r = requests.get(url, params=params, timeout=10)
        r.raise_for_status()
        data = r.json()
        if not data:
            break
        all_results.extend(data)
    return all_results


def safe_get(url: str, headers: dict | None = None, timeout: int = 10) -> dict:
    try:
        r = requests.get(url, headers=headers, timeout=timeout)
        r.raise_for_status()
        return {"status": r.status_code, "data": r.json(), "error": None}
    except requests.exceptions.HTTPError as e:
        return {"status": e.response.status_code, "data": None, "error": str(e)}
    except Exception as e:
        return {"status": None, "data": None, "error": str(e)}


def ai_analyze_results(records: list[dict], question: str, model: str = "llama3.2") -> str:
    summary = json.dumps(records[:5], indent=2)
    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You are an API data analyst. Answer questions about the provided JSON data concisely.",
            },
            {
                "role": "user",
                "content": f"Data (first 5 records):\n{summary}\n\nQuestion: {question}",
            },
        ],
    )
    return response["message"]["content"]

## Your Implementation

Implement `PostsAPI` using the helper functions above. Use `requests.Session()` for connection reuse and persistent auth headers.

In [ ]:
JPH_BASE = "https://jsonplaceholder.typicode.com"


class PostsAPI:
    BASE = JPH_BASE

    def __init__(self, api_key: str = 'demo'):
        # TODO: self.session = requests.Session()
        # TODO: self.session.headers.update(build_headers(api_key))
        pass

    def get_posts(self, user_id: int | None = None, limit: int = 10) -> list[dict]:
        # TODO: build params dict {'_limit': limit} + optionally {'userId': user_id}
        # TODO: self.session.get(url, params=params, timeout=10).raise_for_status()
        # TODO: return response.json()
        pass

    def get_pages(self, page_size: int = 5, max_pages: int = 3) -> list[dict]:
        # TODO: return paginate_collect(f'{self.BASE}/posts', page_size, max_pages)
        pass

    def ai_summary(self, question: str, limit: int = 5) -> str:
        # TODO: posts = self.get_posts(limit=limit)
        # TODO: return ai_analyze_results(posts, question)
        pass

    def safe_fetch(self, endpoint: str) -> dict:
        # TODO: return safe_get(f'{self.BASE}/{endpoint}')
        pass

## Use Your Client

In [ ]:
# 1. Create the client
# client = PostsAPI()

# 2. Fetch 5 posts and print their titles
# posts = client.get_posts(limit=5)
# for p in posts:
#     print(f"  [{p['id']}] {p['title']}")


In [ ]:
# 3. Filter to user 1's posts
# user_posts = client.get_posts(user_id=1, limit=3)
# print(f'User 1 has {len(user_posts)} posts (limited to 3)')


In [ ]:
# 4. Collect 2 pages of 5
# pages = client.get_pages(page_size=5, max_pages=2)
# print(f'Paginated: {len(pages)} total posts')


In [ ]:
# 5. AI summary
# summary = client.ai_summary('What topics or themes appear in these posts?')
# print('AI Summary:')
# print(summary)


## Checks

In [ ]:
def _run_project_checks():
    total = 5
    passed = 0

    # Check 1: PostsAPI class defined with required methods
    try:
        assert 'PostsAPI' in globals(), 'PostsAPI not defined'
        for method in ('get_posts', 'get_pages', 'ai_summary', 'safe_fetch'):
            assert hasattr(PostsAPI, method), f'PostsAPI missing method: {method}'
        passed += 1; print('\u2705 Check 1: PostsAPI class has all required methods')
    except Exception as e:
        print(f'\u274c Check 1: {e}')

    # Check 2: client is a PostsAPI instance
    try:
        assert 'client' in globals(), 'client not defined'
        assert isinstance(client, PostsAPI), \
            f'client should be PostsAPI, got {type(client)}'
        passed += 1; print('\u2705 Check 2: client is a PostsAPI instance')
    except Exception as e:
        print(f'\u274c Check 2: {e}')

    # Check 3: posts is a list with dicts containing 'id' and 'title'
    try:
        assert 'posts' in globals(), 'posts not defined'
        assert isinstance(posts, list) and len(posts) >= 1, \
            f'posts should be non-empty list, got {posts!r}'
        assert 'id' in posts[0] and 'title' in posts[0], \
            f"posts items missing 'id'/'title': {list(posts[0])}"
        passed += 1; print(f'\u2705 Check 3: posts has {len(posts)} items with id/title')
    except Exception as e:
        print(f'\u274c Check 3: {e}')

    # Check 4: pages has more items than a single page
    try:
        assert 'pages' in globals(), 'pages not defined'
        assert isinstance(pages, list), f'pages must be list, got {type(pages)}'
        assert len(pages) >= 6, \
            f'pages should have >= 6 items (at least 2 pages), got {len(pages)}'
        passed += 1; print(f'\u2705 Check 4: pages has {len(pages)} items (multiple pages)')
    except Exception as e:
        print(f'\u274c Check 4: {e}')

    # Check 5: summary is a non-empty string
    try:
        assert 'summary' in globals(), 'summary not defined'
        assert isinstance(summary, str) and len(summary) > 10, \
            f'summary should be non-empty string, got {summary!r}'
        passed += 1; print(f'\u2705 Check 5: summary is a {len(summary)}-char string')
    except Exception as e:
        print(f'\u274c Check 5: {e}')

    if passed == total:
        print('\U0001f389 Project complete!')
    print(f'\nScore: {passed}/{total}')


_run_project_checks()

## Bonus Challenges

- Add `get_comments(post_id)` that fetches `/posts/{id}/comments` using the session
- Add retry logic inside `get_posts`: if the response status is 429, wait and retry
- Replace `PostsAPI.BASE` with a real public API you care about (e.g. Open Library, REST Countries, PokeAPI)
- Use `safe_fetch` to gracefully handle a non-existent endpoint and log the error to a file
- Extend `ai_summary` to accept a `user_id` filter and summarise only that user's posts